# Comparación de Modelos Númericos con puntos alazar

A continución, se muestra la comparación de varios modelos númericos considerando 15 puntos alazar para las siguiente funciones:
- Función Esfera (10 variables)
- Función McCormick (2 variables)
- Función de Dixon-Price (4 variables)
- Función de Easom (2 variables)
- Función de Rastrigin (5 variables)

NOTA:
-  Las funciones y los modelos fueron implementados como modulos individuales para poder ser invocados desde el notebook

## Importación de Dependencias y Generación de puntos aleatorios

In [166]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [167]:
%autoreload 2

In [168]:
# Dependencias

import numpy as np
import random
from config.logging import setUpLogging
from graph.graph_foo import FunctionPlotter2D
from numerical_methods.gradient_descent import MDG_Wolfe
from numerical_methods.newton import NewtonMethod
from numerical_methods.max_decrease import MDG_DM
from functions.sphere_foo import SphereFunction
from functions.mcCormick_foo import McCormickFunction
from functions.dixon_price_foo import DixonPriceFunction
from functions.easom_foo import EasomFunction
from functions.rastrigin_foo import RastriginFunction
from utils.aux_functions import getTestValues, mergeLists, crear_tabla_comparativa, resumen_tabla
from graph.graph_foo import FunctionPlotter2D
import logging

setUpLogging()
logger = logging.getLogger(__name__)

# Cantidad de puntos
n_points = 15

# Función Esfera con 10 variables
sphere = SphereFunction()
test_data_sphere: np.array = getTestValues(n_points, 10, sphere.domain[0], sphere.domain[1])
print("Sphere Test Data: ", test_data_sphere)


# Función McCormick con 2 variables
mcCormick = McCormickFunction()

test_data_mcCormick_x0: np.array = getTestValues(n_points, 1, mcCormick.domain[0], mcCormick.domain[1])
test_data_mcCormick_x1: np.array = getTestValues(n_points, 1, mcCormick.domain_x2[0], mcCormick.domain_x2[1])
test_data_mcCormick: np.array[tuple] = mergeLists(test_data_mcCormick_x0, test_data_mcCormick_x1)
print("McCormick Test Data: ",test_data_mcCormick)

# Función de Dixon-Price con 4 variables
dixon_price = DixonPriceFunction()

test_data_dixon_price: np.array = getTestValues(n_points, 4, dixon_price.domain[0], dixon_price.domain[1])
print("Dixon-Price Test Data: ", test_data_dixon_price)

# Función de Easom con 2 variables
easom = EasomFunction()

test_data_easom: np.array = getTestValues(n_points, 2, easom.domain[0], easom.domain[1])
print("Easom Test Data: ", test_data_easom)

# Función de Rastrigin con 5 variables
rastrigin = RastriginFunction()

test_data_rastrigin: np.array = getTestValues(n_points, 5, rastrigin.domain[0], rastrigin.domain[1])
print("Rastrigin Test Data: ", test_data_rastrigin)



Sphere Test Data:  [[-8.27071677e-01  1.35268037e+00  4.79484706e-01  3.42358747e+00
   4.76951059e+00  2.11343386e+00  3.97407964e+00 -4.58287827e+00
  -3.01021865e+00 -2.22297965e+00]
 [-1.48061601e-01  5.76938089e-01  2.25146783e+00  1.82342141e-01
   3.98317049e+00  1.15572064e+00  1.99916449e-01 -3.78133191e+00
  -3.97722024e+00 -2.44476669e+00]
 [-7.13435904e-01  2.59173306e+00 -1.67015426e+00 -2.07069672e+00
  -2.55357970e+00 -1.13167725e+00 -4.19289092e+00 -3.87131867e+00
  -2.84952827e+00 -1.53027151e+00]
 [ 6.99454374e-01  1.53582627e+00 -4.74895324e-02  3.43618370e+00
   4.24093076e+00  4.54572485e+00 -4.78490404e+00 -3.68864683e+00
  -4.97758208e+00 -3.52194965e+00]
 [-9.92657158e-01  2.00171876e+00  2.10078948e+00  4.81946315e+00
   4.15328898e+00  2.62224069e+00  3.05332476e+00  5.04135574e+00
   4.79450340e+00 -4.80564996e+00]
 [-9.94037001e-01  4.28095001e-02  2.09898054e+00  2.89355415e+00
   4.25039222e+00 -4.06073820e+00 -2.30291736e+00 -4.65590878e+00
  -2.31878381e

# Comparación: Función Esfera (10 variables)

In [169]:

# Descenso del Gradiente con condiciones fuertes de Wolfe

resultados_sphere_MDG_Wolfe = {}
resultados_sphere_Newton = {}
resultados_sphere_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_sphere):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = sphere.f,
        Df       = sphere.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_sphere_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        "llego_optimo": np.linalg.norm(x_opt_1 - np.array(sphere.global_min)) < sphere.tol,
        "dist": np.linalg.norm(x_opt_1 - np.array(sphere.global_min)),
        "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": sphere.f_count_invok,
        "Df_invok": sphere.Df_count_invok,
        "H_invok": sphere.H_count_invok
    }

    sphere.f_count_invok = 0
    sphere.Df_count_invok = 0
    sphere.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=sphere.Df,
        H=sphere.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_sphere_Newton[i] = {
        "x_optimo": x_opt_2,
        "llego_optimo": np.linalg.norm(x_opt_2 - np.array(sphere.global_min)) < sphere.tol,
        "dist": np.linalg.norm(x_opt_2 - np.array(sphere.global_min)),
        "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": sphere.f_count_invok,
        "Df_invok": sphere.Df_count_invok,
        "H_invok": sphere.H_count_invok
    }

    sphere.f_count_invok = 0
    sphere.Df_count_invok = 0
    sphere.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=sphere.f,
        Df=sphere.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_sphere_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        "llego_optimo": np.linalg.norm(x_opt_3 - np.array(sphere.global_min)) < sphere.tol,
        "dist": np.linalg.norm(x_opt_3 - np.array(sphere.global_min)),
        "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": sphere.f_count_invok,
        "Df_invok": sphere.Df_count_invok,
        "H_invok": sphere.H_count_invok
    }

    sphere.f_count_invok = 0
    sphere.Df_count_invok = 0
    sphere.H_count_invok = 0

tabla_sphere = crear_tabla_comparativa(
    resultados_por_metodo={
        "MDG_Wolfe": resultados_sphere_MDG_Wolfe,
        "Newton":    resultados_sphere_Newton,
        "MDG_DM":    resultados_sphere_MDG_DM,
    },
    nombre_funcion="Sphere"
)

resumen_sphere = resumen_tabla(tabla_sphere)

display(tabla_sphere)
display(resumen_sphere)





📊 Tabla comparativa — Sphere
📋 Resumen por método


,Punto,Método,x_optimo,llego_optimo,dist,n_iter,f_invok,Df_invok,H_invok
0,1,MDG_DM,"[-0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0....",True,9.616580e-80,20,320,340,0
1,1,MDG_Wolfe,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,0.000000e+00,20,42,60,0
2,1,Newton,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,0.000000e+00,20,0,2,2
3,2,MDG_DM,"[-0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0....",True,7.667599e-80,20,320,340,0
4,2,MDG_Wolfe,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,0.000000e+00,20,42,60,0
5,2,Newton,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,0.000000e+00,20,0,2,2
6,3,MDG_DM,"[-0.0, 0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0...",True,8.070546e-80,20,320,340,0
7,3,MDG_Wolfe,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,0.000000e+00,20,42,60,0
8,3,Newton,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,0.000000e+00,20,0,2,2
9,4,MDG_DM,"[0.0, 0.0, -0.0, 0.0, 0.0, 0.0, -0.0, -0.0, -0...",True,1.126892e-79,20,320,340,0


,Método,Llegó al óptimo,Mejor dist,Mejor x_optimo,Avg f_invok,Avg Df_invok,Avg H_invok
0,MDG_DM,15/15,0.0,"[-0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0....",320.0,340.0,0.0
1,MDG_Wolfe,15/15,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",42.0,60.0,0.0
2,Newton,15/15,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,2.0,2.0


# Comparación: Función McCormick (2 variables)

In [170]:
# Descenso del Gradiente con condiciones fuertes de Wolfe

resultados_mcCormick_MDG_Wolfe = {}
resultados_mcCormick_Newton = {}
resultados_mcCormick_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_mcCormick):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = mcCormick.f,
        Df       = mcCormick.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_mcCormick_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        "llego_optimo": np.linalg.norm(x_opt_1 - np.array(mcCormick.global_min)) < mcCormick.tol,
        "dist": np.linalg.norm(x_opt_1 - np.array(mcCormick.global_min)),
        "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": mcCormick.f_count_invok,
        "Df_invok": mcCormick.Df_count_invok,
        "H_invok": mcCormick.H_count_invok
    }

    mcCormick.f_count_invok = 0
    mcCormick.Df_count_invok = 0
    mcCormick.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=mcCormick.Df,
        H=mcCormick.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_mcCormick_Newton[i] = {
        "x_optimo": x_opt_2,
        "llego_optimo": np.linalg.norm(x_opt_2 - np.array(mcCormick.global_min)) < mcCormick.tol,
        "dist": np.linalg.norm(x_opt_2 - np.array(mcCormick.global_min)),
        "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": mcCormick.f_count_invok,
        "Df_invok": mcCormick.Df_count_invok,
        "H_invok": mcCormick.H_count_invok
    }

    mcCormick.f_count_invok = 0
    mcCormick.Df_count_invok = 0
    mcCormick.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=mcCormick.f,
        Df=mcCormick.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_mcCormick_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        "llego_optimo": np.linalg.norm(x_opt_3 - np.array(mcCormick.global_min)) < mcCormick.tol,
        "dist": np.linalg.norm(x_opt_3 - np.array(mcCormick.global_min)),
        "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": mcCormick.f_count_invok,
        "Df_invok": mcCormick.Df_count_invok,
        "H_invok": mcCormick.H_count_invok
    }

    mcCormick.f_count_invok = 0
    mcCormick.Df_count_invok = 0
    mcCormick.H_count_invok = 0

tabla_mcCormick = crear_tabla_comparativa(
    resultados_por_metodo={
        "MDG_Wolfe": resultados_mcCormick_MDG_Wolfe,
        "Newton":    resultados_mcCormick_Newton,
        "MDG_DM":    resultados_mcCormick_MDG_DM,
    },
    nombre_funcion="McCormick"
)

resumen_mcCormick = resumen_tabla(tabla_mcCormick)


display(tabla_mcCormick)
display(resumen_mcCormick)


📊 Tabla comparativa — McCormick
📋 Resumen por método


,Punto,Método,x_optimo,llego_optimo,dist,n_iter,f_invok,Df_invok,H_invok
0,1,MDG_DM,"[5.7736, 4.7501]",False,8.922344,20,368,388,0
1,1,MDG_Wolfe,"[7.119, 4.403]",False,9.704442,20,1089,44,0
2,1,Newton,"[793.1754, 775.5844]",False,1110.823587,20,0,20,20
3,2,MDG_DM,"[2.6993, 1.6288]",False,4.541587,20,392,412,0
4,2,MDG_Wolfe,"[3.3824, 0.7726]",False,4.563280,20,1032,42,0
5,2,Newton,"[1285.3816, 1267.7717]",False,1806.871136,20,0,20,20
6,3,MDG_DM,"[11.8361, 10.9241]",False,17.574943,20,480,500,0
7,3,MDG_Wolfe,"[1.9726, -2.5499]",False,2.711944,20,1111,49,0
8,3,Newton,"[110.319, 85.9528]",False,141.235804,20,0,20,20
9,4,MDG_DM,"[3.6806, 1.4579]",False,5.187003,20,552,572,0


,Método,Llegó al óptimo,Mejor dist,Mejor x_optimo,Avg f_invok,Avg Df_invok,Avg H_invok
0,MDG_DM,0/15,—,None,427.73,447.73,0.0
1,MDG_Wolfe,0/15,—,None,1058.20,44.00,0.0
2,Newton,0/15,—,None,0.00,20.00,20.0


# Comparación: Función de Dixon-Price (4 variables)

In [171]:

resultados_dixon_price_MDG_Wolfe = {}
resultados_dixon_price_Newton = {}
resultados_dixon_price_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_dixon_price):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = dixon_price.f,
        Df       = dixon_price.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_dixon_price_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        "llego_optimo": np.linalg.norm(x_opt_1 - np.array(dixon_price.global_min)) < dixon_price.tol,
        "dist": np.linalg.norm(x_opt_1 - np.array(dixon_price.global_min)),
        "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": dixon_price.f_count_invok,
        "Df_invok": dixon_price.Df_count_invok,
        "H_invok": dixon_price.H_count_invok
    }

    dixon_price.f_count_invok = 0
    dixon_price.Df_count_invok = 0
    dixon_price.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=dixon_price.Df,
        H=dixon_price.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_dixon_price_Newton[i] = {
        "x_optimo": x_opt_2,
        "llego_optimo": np.linalg.norm(x_opt_2 - np.array(dixon_price.global_min)) < dixon_price.tol,
        "dist": np.linalg.norm(x_opt_2 - np.array(dixon_price.global_min)),
        "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": dixon_price.f_count_invok,
        "Df_invok": dixon_price.Df_count_invok,
        "H_invok": dixon_price.H_count_invok
    }

    dixon_price.f_count_invok = 0
    dixon_price.Df_count_invok = 0
    dixon_price.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=dixon_price.f,
        Df=dixon_price.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_dixon_price_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        "llego_optimo": np.linalg.norm(x_opt_3 - np.array(dixon_price.global_min)) < dixon_price.tol,
        "dist": np.linalg.norm(x_opt_3 - np.array(dixon_price.global_min)),
        "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": dixon_price.f_count_invok,
        "Df_invok": dixon_price.Df_count_invok,
        "H_invok": dixon_price.H_count_invok
    }

    dixon_price.f_count_invok = 0
    dixon_price.Df_count_invok = 0
    dixon_price.H_count_invok = 0

tabla_dixon_price = crear_tabla_comparativa(
    resultados_por_metodo={
        "MDG_Wolfe": resultados_dixon_price_MDG_Wolfe,
        "Newton":    resultados_dixon_price_Newton,
        "MDG_DM":    resultados_dixon_price_MDG_DM,
    },
    nombre_funcion="Dixon-Price"
)

resumen_dixon_price = resumen_tabla(tabla_dixon_price)

display(tabla_dixon_price)
display(resumen_dixon_price)



📊 Tabla comparativa — Dixon-Price
📋 Resumen por método


,Punto,Método,x_optimo,llego_optimo,dist,n_iter,f_invok,Df_invok,H_invok
0,1,MDG_DM,"[-6.3402, -5.9945, -4.4631, -2.2255]",False,11.491237,20,320,340,0
1,1,MDG_Wolfe,"[0.3331, 0.0009, 0.001, -0.0222]",False,1.271963,20,123,60,0
2,1,Newton,"[0.333, 0.0044, -0.0, -0.0]",False,1.260771,20,0,20,20
3,2,MDG_DM,"[8.3876, -5.5641, 6.1139, -1.3638]",False,11.314256,20,320,340,0
4,2,MDG_Wolfe,"[1.8999, 0.9608, 0.7424, -0.5335]",False,1.435153,20,168,61,0
5,2,Newton,"[0.3324, -0.0532, -0.0935, 0.2239]",False,1.265121,20,0,20,20
6,3,MDG_DM,"[-1.9639, 4.8153, 5.9599, -9.679]",False,12.608850,20,320,340,0
7,3,MDG_Wolfe,"[0.477, 0.3754, 0.4248, -0.4059]",False,1.147689,20,154,63,0
8,3,Newton,"[1.0116, 0.7132, 0.5972, -0.5464]",False,1.091783,20,0,20,20
9,4,MDG_DM,"[-6.893, -5.9632, -9.5735, 7.7649]",False,16.195870,20,320,340,0


,Método,Llegó al óptimo,Mejor dist,Mejor x_optimo,Avg f_invok,Avg Df_invok,Avg H_invok
0,MDG_DM,0/15,—,None,320.00,340.00,0.0
1,MDG_Wolfe,0/15,—,None,155.73,61.07,0.0
2,Newton,0/15,—,None,0.00,20.00,20.0


# Comparación: Función de Easom (2 variables)

In [172]:

resultados_easom_MDG_Wolfe = {}
resultados_easom_Newton = {}
resultados_easom_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_easom):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = easom.f,
        Df       = easom.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_easom_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        "llego_optimo": np.linalg.norm(x_opt_1 - np.array(easom.global_min)) < easom.tol,
        "dist": np.linalg.norm(x_opt_1 - np.array(easom.global_min)),
        "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": easom.f_count_invok,
        "Df_invok": easom.Df_count_invok,
        "H_invok": easom.H_count_invok
    }

    easom.f_count_invok = 0
    easom.Df_count_invok = 0
    easom.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=easom.Df,
        H=easom.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_easom_Newton[i] = {
        "x_optimo": x_opt_2,
        "llego_optimo": np.linalg.norm(x_opt_2 - np.array(easom.global_min)) < easom.tol,
        "dist": np.linalg.norm(x_opt_2 - np.array(easom.global_min)),
        "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": easom.f_count_invok,
        "Df_invok": easom.Df_count_invok,
        "H_invok": easom.H_count_invok
    }

    easom.f_count_invok = 0
    easom.Df_count_invok = 0
    easom.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=easom.f,
        Df=easom.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_easom_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        "llego_optimo": np.linalg.norm(x_opt_3 - np.array(easom.global_min)) < easom.tol,
        "dist": np.linalg.norm(x_opt_3 - np.array(easom.global_min)),
        "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": easom.f_count_invok,
        "Df_invok": easom.Df_count_invok,
        "H_invok": easom.H_count_invok
    }

    easom.f_count_invok = 0
    easom.Df_count_invok = 0
    easom.H_count_invok = 0

tabla_easom = crear_tabla_comparativa(
    resultados_por_metodo={
        "MDG_Wolfe": resultados_easom_MDG_Wolfe,
        "Newton":    resultados_easom_Newton,
        "MDG_DM":    resultados_easom_MDG_DM,
    },
    nombre_funcion="Easom"
)

resumen_easom = resumen_tabla(tabla_easom)

display(tabla_easom)
display(resumen_easom)


📊 Tabla comparativa — Easom
📋 Resumen por método


/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/numerical_methods/max_decrease.py:20: RuntimeWarning: invalid value encountered in scalar divide
  xk1 = (df(xk1)*xk - df(xk)*xk1) / (df(xk1) - df(xk))
/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/numerical_methods/max_decrease.py:20: RuntimeWarning: divide by zero encountered in scalar divide
  xk1 = (df(xk1)*xk - df(xk)*xk1) / (df(xk1) - df(xk))
/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/functions/easom_foo.py:27: RuntimeWarning: invalid value encountered in cos
  return -1*cos(x[0])*cos(x[1])*exp(-1*(x[0] - pi)**2 - (x[1] - pi)**2)


,Punto,Método,x_optimo,llego_optimo,dist,n_iter,f_invok,Df_invok,H_invok
0,1,MDG_DM,"[nan, nan]",False,NaN,20,3200,3220,0
1,1,MDG_Wolfe,"[-4.1848, 5.5796]",False,7.721380e+00,20,2080,60,0
2,1,Newton,"[2.7839, -10.0486]",False,1.319508e+01,20,0,20,20
3,2,MDG_DM,"[nan, nan]",False,NaN,20,3200,3220,0
4,2,MDG_Wolfe,"[5.4907, -1.4053]",False,5.117886e+00,20,2080,60,0
5,2,Newton,"[2.7812, -0.8777]",False,4.035395e+00,20,0,20,20
6,3,MDG_DM,"[nan, nan]",False,NaN,20,3048,3068,0
7,3,MDG_Wolfe,"[0.8741, 6.2671]",False,3.861389e+00,20,2080,60,0
8,3,Newton,"[-17.5915, 9.674]",False,2.173782e+01,20,0,20,20
9,4,MDG_DM,"[nan, nan]",False,NaN,20,3200,3220,0


,Método,Llegó al óptimo,Mejor dist,Mejor x_optimo,Avg f_invok,Avg Df_invok,Avg H_invok
0,MDG_DM,0/15,—,None,2996.27,3016.27,0.00
1,MDG_Wolfe,2/15,0.0,"[3.1416, 3.1416]",1859.87,60.00,0.00
2,Newton,1/15,0.000027,"[3.1416, 3.1416]",0.00,19.53,19.53


# Comparación: Función de Rastrigin (5 variables)

In [173]:

resultados_rastrigin_MDG_Wolfe = {}
resultados_rastrigin_Newton = {}
resultados_rastrigin_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_rastrigin):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = rastrigin.f,
        Df       = rastrigin.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_rastrigin_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        "llego_optimo": np.linalg.norm(x_opt_1 - np.array(rastrigin.global_min)) < rastrigin.tol,
        "dist": np.linalg.norm(x_opt_1 - np.array(rastrigin.global_min)),
        "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": rastrigin.f_count_invok,
        "Df_invok": rastrigin.Df_count_invok,
        "H_invok": rastrigin.H_count_invok
    }

    rastrigin.f_count_invok = 0
    rastrigin.Df_count_invok = 0
    rastrigin.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=rastrigin.Df,
        H=rastrigin.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_rastrigin_Newton[i] = {
        "x_optimo": x_opt_2,
        "llego_optimo": np.linalg.norm(x_opt_2 - np.array(rastrigin.global_min)) < rastrigin.tol,
        "dist": np.linalg.norm(x_opt_2 - np.array(rastrigin.global_min)),
        "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": rastrigin.f_count_invok,
        "Df_invok": rastrigin.Df_count_invok,
        "H_invok": rastrigin.H_count_invok
    }

    rastrigin.f_count_invok = 0
    rastrigin.Df_count_invok = 0
    rastrigin.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=rastrigin.f,
        Df=rastrigin.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_rastrigin_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        "llego_optimo": np.linalg.norm(x_opt_3 - np.array(rastrigin.global_min)) < rastrigin.tol,
        "dist": np.linalg.norm(x_opt_3 - np.array(rastrigin.global_min)),
        "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": rastrigin.f_count_invok,
        "Df_invok": rastrigin.Df_count_invok,
        "H_invok": rastrigin.H_count_invok
    }

    rastrigin.f_count_invok = 0
    rastrigin.Df_count_invok = 0
    rastrigin.H_count_invok = 0

tabla_rastrigin = crear_tabla_comparativa(
    resultados_por_metodo={
        "MDG_Wolfe": resultados_rastrigin_MDG_Wolfe,
        "Newton":    resultados_rastrigin_Newton,
        "MDG_DM":    resultados_rastrigin_MDG_DM,
    },
    nombre_funcion="Rastrigin"
)

resumen_rastrigin = resumen_tabla(tabla_rastrigin)

display(tabla_rastrigin)  # display() renderiza mejor que print en notebooks
display(resumen_rastrigin)



📊 Tabla comparativa — Rastrigin
📋 Resumen por método


,Punto,Método,x_optimo,llego_optimo,dist,n_iter,f_invok,Df_invok,H_invok
0,1,MDG_DM,"[-15.4357, -6.9106, -16.5068, -4.6669, 12.7444]",False,27.252352,20,2200,2220,0
1,1,MDG_Wolfe,"[0.9947, -0.0002, 0.9948, -0.0001, -2.985]",False,3.299877,20,428,61,0
2,1,Newton,"[1.9899, 3.9798, 3.5179, 4.523, -0.995]",False,7.322648,20,0,4,4
3,2,MDG_DM,"[17.4236, -2.0794, -6.6338, -21.8268, -2.2918]",False,28.871710,20,2320,2340,0
4,2,MDG_Wolfe,"[-1.9872, 1.989, 1.9914, -3.9797, 0.0025]",False,5.263885,20,810,54,0
5,2,Newton,"[-0.995, 3.5179, 3.9798, -3.5179, -0.995]",False,6.524510,20,0,4,4
6,3,MDG_DM,"[0.6441, -8.4557, 6.0458, -7.424, 10.6428]",False,16.638888,20,2216,2236,0
7,3,MDG_Wolfe,"[-0.0006, -0.0009, 2.9847, -3.9801, -1.9891]",False,5.357771,20,740,50,0
8,3,Newton,"[-1.5076, 0.0, 2.9849, -3.5179, -1.9899]",False,5.245706,20,0,6,6
9,4,MDG_DM,"[2.9894, 13.7586, 2.9888, 7.5296, -4.8804]",False,16.961191,20,2280,2300,0


,Método,Llegó al óptimo,Mejor dist,Mejor x_optimo,Avg f_invok,Avg Df_invok,Avg H_invok
0,MDG_DM,0/15,—,None,2286.93,2306.93,0.0
1,MDG_Wolfe,0/15,—,None,759.67,52.20,0.0
2,Newton,0/15,—,None,0.00,5.80,5.8
